# Módulo 4 — Dependencia temporal y diagnóstico de series

**Curso: Análisis y Pronóstico de Datos Mineros con Python**

> ACF/PACF, estacionariedad, ADF/KPSS, transformaciones y diferenciación: dejar la serie lista para modelar.

---

### Cómo usar este notebook
1. Ábrelo en **Google Colab** y ejecuta las celdas **de arriba hacia abajo**.
2. Los datos se descargan solos desde el repositorio del curso; no tienes que subir nada.
3. Lee las salidas: cada bloque responde una pregunta concreta, no ejecutes por ejecutar.

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns          # gráficos estadísticos (preinstalado en Colab)

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.width', 120)

# --- Datos del curso -------------------------------------------------
# Los CSV viven en el repositorio del curso y se descargan solos.
# Si no hubiera internet, la función pide subir el archivo a mano.
REPO_DATOS = 'https://raw.githubusercontent.com/HishanFarfan/curso-datos-mineros/main/datos'

def cargar_datos(nombre, **kw):
    try:
        return pd.read_csv(f'{REPO_DATOS}/{nombre}', **kw)
    except Exception as e:
        print('No se pudo descargar desde GitHub:', e)
    try:
        from google.colab import files          # Colab: subir a mano
        print(f"Sube '{nombre}':")
        return pd.read_csv(next(iter(files.upload())), **kw)
    except ModuleNotFoundError:
        return pd.read_csv(nombre, **kw)         # local: archivo en el cwd

## 1. Carga de la serie preparada

Partimos de la versión ya limpia y ordenada (`_LIMPIO.csv`). En un flujo real usarías la salida del Módulo 1.

In [ ]:
df = cargar_datos('datos_proceso_planta_LIMPIO.csv', parse_dates=['Fecha'])
df = df.sort_values('Fecha').set_index('Fecha')
df = df.asfreq('h')   # eje horario regular; expone huecos como NaN
df.head()

In [ ]:
serie = df['Recuperacion_pct'].dropna()
serie.plot(title='Recuperación'); plt.show()

## 2. ACF y PACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(serie, lags=60, ax=ax[0])
plot_pacf(serie, lags=60, ax=ax[1], method='ywm')
plt.show()

## 3. Diagnóstico visual de estacionariedad

Nivel (~85–90) y dispersión (~2–5) están en escalas muy distintas: en un mismo eje la caída de la media no se aprecia. Un panel por estadístico.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
serie.rolling(168).mean().plot(ax=ax[0], color='tab:blue')
ax[0].set(title='Media móvil (semana) — ¿cambia el nivel?', ylabel='Recuperación [%]')
serie.rolling(168).std().plot(ax=ax[1], color='tab:orange')
ax[1].set(title='Std móvil (semana) — ¿cambia la dispersión?', ylabel='desv. est. [%]')
plt.tight_layout(); plt.show()

## 4. Pruebas ADF y KPSS

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def diagnostico(x, nombre=''):
    x = pd.Series(x).dropna()
    p_adf = adfuller(x)[1]
    import warnings; warnings.filterwarnings('ignore')
    p_kpss = kpss(x, regression='c', nlags='auto')[1]
    print(f'{nombre:>22} | ADF p={p_adf:6.3f} ({"estacionaria" if p_adf<0.05 else "raíz unitaria"})'
          f' | KPSS p={p_kpss:6.3f} ({"no estac." if p_kpss<0.05 else "estac."})')

diagnostico(serie, 'Recuperación (nivel)')

## 5. Transformaciones

In [ ]:
diagnostico(serie.diff(), 'Δ Recuperación')
diagnostico(np.log(df['Tonelaje_tph'].dropna()), 'log Tonelaje')
diagnostico(np.log(df['Tonelaje_tph'].dropna()).diff(), 'Δ log Tonelaje')
diagnostico(df['Tonelaje_tph'].dropna().diff().diff(24), 'Δ Δ24 Tonelaje')

In [ ]:
# Comparar ACF antes y después de diferenciar
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(serie, lags=60, ax=ax[0], title='ACF nivel')
plot_acf(serie.diff().dropna(), lags=60, ax=ax[1], title='ACF Δ')
plt.show()

## 6. Hoja de decisión (complétala)

| Característica | Recuperación | Tonelaje |
|---|---|---|
| ¿Tendencia? | | |
| ¿Estacionalidad? (m) | | |
| ¿Varianza estable? | | |
| ¿ACF persistente? | | |
| ADF / KPSS | | |
| Transformación | | |
| d | | |
| D , m | | |

## Actividades sugeridas

1. Ejecuta `diagnostico` sobre `Ley_Cu_pct` en nivel y en primera diferencia.
2. ¿La diferenciación estacional (24) elimina los picos de la ACF en 24, 48, 72?
3. Prueba `scipy.stats.boxcox` sobre el tonelaje y compara con `np.log`.
4. Escribe en una frase la decisión final de preparación para cada variable.

---
## Cierre

La serie queda con una decisión documentada (transformación + d + D + m). El Módulo 5 la usa para construir ARIMA/SARIMA.